In [4]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
import pypsa
import xlsxwriter
import tz_pypsa
import tz_pypsa.wrangle as wrangle
import pandas as pd
# import tz_solve
import plotly.express as px
import plotly.graph_objects as go
from tz_pypsa.model import Model
from tz_pypsa.utils import get_examples
import os
import glob

In [16]:
n_tokyo = pypsa.Network()
n_tokyo.import_from_netcdf("C:/Users/jy/TransitionZero/Google - CFE - Documents/04. Country Specific Vault/Japan/02. Data & Results/Outputs/1_Diagnosis/TP2/Run004/JPN_P2_JPN03_UpdatedStorage/solved_networks/hourly_matching_CFE100_2030.nc")

INFO:pypsa.io:Imported network hourly_matching_CFE100_2030.nc has buses, carriers, generators, links, loads, storage_units


In [3]:
n_kansai = pypsa.Network()
n_kansai.import_from_netcdf("C:/Users/jy/TransitionZero/Google - CFE - Documents/04. Country Specific Vault/Japan/02. Data & Results/Outputs/1_Diagnosis/TP1/Run004/Data/JPN06/solved_networks/hourly_matching_CFE70_2030.nc")

INFO:pypsa.io:Imported network hourly_matching_CFE70_2030.nc has buses, carriers, generators, links, loads, storage_units


In [4]:
n_chubu = pypsa.Network()
n_chubu.import_from_netcdf("C:/Users/jy/TransitionZero/Google - CFE - Documents/04. Country Specific Vault/Japan/02. Data & Results/Outputs/1_Diagnosis/TP1/Run004/Data/JPN04/solved_networks/hourly_matching_CFE70_2030.nc")

INFO:pypsa.io:Imported network hourly_matching_CFE70_2030.nc has buses, carriers, generators, links, loads, storage_units


In [ ]:
n = pypsa.Network()
n.import_from_netcdf("C:/Users/jy/TransitionZero/Google - CFE - Documents/04. Country Specific Vault/Japan/02. Data & Results/Outputs/Brownfield/Tableau/nc_files/Brownfield-013.nc")

INFO:pypsa.io:Imported network Brownfield-013.nc has buses, carriers, generators, links, loads, storage_units


In [26]:
n.links_t.p0.to_csv("reference_scenario_interconnector_flow.csv")

In [28]:
n.links.groupby('bus1').p_nom.sum()

bus1
JPN01     1200.0
JPN02     7310.0
JPN03    11300.0
JPN04     4620.0
JPN05     1900.0
JPN06    10650.0
JPN07     6760.0
JPN08     1200.0
JPN09      620.0
Name: p_nom, dtype: float64

In [3]:
def GetGridCFE(
    n: pypsa.Network,   
    ci_identifier: str,
    bus: str
):
    """

    Calculate the CFE score of a grid, intra- and inter-regionally. Here, we follow the mathematical
    expressions presented by Xu and Jenkins (2021): https://acee.princeton.edu/24-7/

    Parameters:
    -----------
    network : pypsa.Network
        The optimised network for which we are calculating the GridCFE.
    bus : str
        The country bus for which we are calculating the GridCFE.
    ci_identifier : str
        The unique identifer used to identify C&I assets.

    Returns:
    -----------
    CFE Score: list
        Hourly resolution CFE scores for each snapshot in the network.

    Let:
    -----------
    R = Intra-regional grid
    Z = Inter-regional grid

    """

    

        # get global clean carriers
    global_clean_carriers = [
        i
        for i in n.carriers.query(" co2_emissions <= 0").index.tolist()
        if i in n.generators.carrier.tolist()
    ]

    # get clean generators in R
    R_clean_generators = n.generators.loc[
        # clean carriers
        (n.generators.carrier.isin(global_clean_carriers))
        &
        #exclude assets not in R
        (n.generators.index.str.contains(bus)) &
        # exclude C&I assets
        (~n.generators.index.str.contains(ci_identifier))
    ].index

    # get all generators
    R_all_generators = n.generators.loc[
        (~n.generators.index.str.contains(ci_identifier))
        &
        (n.generators.index.str.contains(bus)) 
    ].index

    # calculate CFE sceore
    total_clean_generation = n.generators_t.p[R_clean_generators].sum(axis=1)
    total_generation = n.generators_t.p[R_all_generators].sum(axis=1)

    # return CFE score
    return (total_clean_generation / total_generation).round(2).tolist()

In [10]:
n_tokyo.generators_t.p.filter(regex='C&I')

Generator,JPN03 C&I Grid-gas-CCS-unspecified-ext-2030-PPA-Clean,JPN03 C&I Grid-gas-CCS-unspecified-ext-2030-PPA-Fossil,JPN03 C&I Grid-onshorewind-unspecified-ext-2030-PPA-Clean,JPN03 C&I Grid-solar-unspecified-ext-2030-PPA-Clean
snapshot,,,,
2030-01-01 00:00:00,0.001019,0.000437,1206.105921,0.0
2030-01-01 01:00:00,0.000809,0.000347,1244.762654,0.0
2030-01-01 02:00:00,0.001761,0.000755,1240.897042,0.0
2030-01-01 03:00:00,0.002682,0.001149,1302.748661,0.0
2030-01-01 04:00:00,0.000428,0.000183,1291.151593,0.0
...,...,...,...,...
2030-12-31 19:00:00,0.003358,0.001439,1561.752297,0.0
2030-12-31 20:00:00,0.002847,0.001220,1530.826496,0.0
2030-12-31 21:00:00,0.000650,0.000279,1546.289434,0.0


In [23]:
def get_cfe_score_ts(n, bus, ci_identifier='C&I'):
    '''Calculate the CFE score and return it as a time series
    '''
    GridCFE = GetGridCFE(n, ci_identifier=ci_identifier, bus=bus)
    CI_Demand = n.loads_t.p.filter(regex=ci_identifier).sum(axis=1)
    CI_PPA = n.generators_t.p.filter(regex='C&I').sum(axis=1)
    CI_GridExport = n.links_t.p0.filter(regex='Exports').sum(axis=1)
    CI_GridImport = n.links_t.p0.filter(regex='Imports').sum(axis=1)
    CI_StorageDischarge = n.links_t.p0.filter(regex='Discharge').sum(axis=1)
    CI_StorageCharge = n.links_t.p0.filter(regex='Charge').sum(axis=1)
    
    return (( CI_PPA - CI_GridExport + (CI_GridImport * list(GridCFE) ) - CI_StorageCharge + CI_StorageDischarge ) / CI_Demand).to_frame(name='CFE Score')

In [11]:
n_tokyo.links_t.p0.filter(regex='Exports').sum(axis=1).to_csv('TP1_tokyo_CFE100_exports.csv')

In [17]:
n_tokyo.loads_t.p.filter(regex='C&I').sum(axis=1).sum()

9785312.812

In [31]:
pd.Series(GetGridCFE(n_tokyo, ci_identifier='C&I', bus="JPN03")).to_csv("gridcfescore.csv")

In [28]:
get_cfe_score_ts(n_tokyo, "JPN03", ci_identifier='C&I').to_csv("CFE90_ts_tokyo.csv")

In [7]:
CFEScore_70_ts_tokyo = get_cfe_score_ts(n_tokyo, "JPN03", ci_identifier='C&I')
CFEScore_70_ts_kansai = get_cfe_score_ts(n_kansai, "JPN06", ci_identifier='C&I')
CFEScore_70_ts_chubu = get_cfe_score_ts(n_chubu, "JPN04", ci_identifier='C&I')

In [9]:
CFEScore_70_ts_tokyo.to_csv("CFEScore_70_ts_tokyo.csv")
CFEScore_70_ts_kansai.to_csv("CFEScore_70_ts_kansai.csv")
CFEScore_70_ts_chubu.to_csv("CFEScore_70_ts_chubu.csv")

In [11]:
n_tokyo.loads_t.p.filter(regex='C&I').to_csv("CI_demand_tokyo.csv")
n_kansai.loads_t.p.filter(regex='C&I').to_csv("CI_demand_kansai.csv")
n_chubu.loads_t.p.filter(regex='C&I').to_csv("CI_demand_chubu.csv")

In [ ]:
n_tokyo.link

component    carrier                       
Link         Transmission                      1995.62500
Generator    Biomass                              0.00184
             Coal                                 0.00000
             Coal & blue NH3: Coal                0.00148
             Coal & blue NH3: NH3                 0.00131
             Gas                                  0.00512
             Gas & blue H2: Gas                   0.00043
             Gas & blue H2: H2                    0.00047
             Gas CCS: Leaked Component            0.00030
             Gas CCS: Sequestered component       0.00028
             Geothermal                           0.00205
             Hydro conventional                   0.00000
             Nuclear                              0.00000
             Offshore Wind                        0.00000
             Oil                                  0.00000
             Onshore Wind                         0.00680
             Solar          

In [9]:
# Define your mapping
generator_tariff_mapping = {
    'TWN C&I Grid-solar-unspecified-ext-2030-PPA': 12,
    'TWN C&I Grid-onshorewind-unspecified-ext-2030-PPA': 8,
}

# Add the column with default values
df = pd.DataFrame(index=n.generators.index, columns=['feed_in_tariff'])
df['feed_in_tariff'] = 0.0
n.add('Generator', df.index, **df)

# Apply the mapping
for gen_name, tariff in generator_tariff_mapping.items():
    if gen_name in n.generators.index:
        n.generators.feed_in_tariff.loc[gen_name] = tariff


Index(['TWN-Coal-2024active_exo', 'TWN-Coal-2030_exo',
       'TWN-LNG-2024active_exo', 'TWN-LNG-2030_endo', 'TWN-LNG-2030_exo',
       'TWN-Oil-2024active_exo', 'TWN-Oil-2030_exo',
       'TWN-Nuclear-2024active_exo', 'TWN-Nuclear-2030_exo',
       'TWN-Hydro-2024active_exo', 'TWN-Hydro-2030_exo',
       'TWN-Biomass-2024active_exo', 'TWN-Biomass-2030_endo',
       'TWN-Biomass-2030_exo', 'TWN-Geothermal-2024active_exo',
       'TWN-Geothermal-2030_endo', 'TWN-Geothermal-2030_exo',
       'TWN-Solar-2024active_exo', 'TWN-Solar-2030_endo', 'TWN-Solar-2030_exo',
       'TWN-OffshoreWind-2024active_exo', 'TWN-OffshoreWind-2030_endo',
       'TWN-OffshoreWind-2030_exo', 'TWN-OnshoreWind-2024active_exo',
       'TWN-OnshoreWind-2030_endo', 'TWN-OnshoreWind-2030_exo',
       'TWN-CoGen_coal-2024active_exo', 'TWN-CoGen_coal-2030_exo',
       'TWN-CoGen_biogas_waste-2024active_exo',
       'TWN-CoGen_biogas_waste-2030_exo', 'TWN-BlueH2-2030_exo',
       'TWN-BlueH2_Gas-2030_exo', 'TWN-BlueNH3

In [10]:
wrangle.calculate_weighted_feed_in_tariff(n)

8.000000000043826

In [12]:
df = wrangle.get_ci_cost_summary(n)

In [13]:
df.to_csv("data.csv")